In [6]:
import pandas as pd
from pathlib import Path
import numpy as np

In [2]:
def adjust_df_display(dimension, action):
    """This function when called adjusts the output display of pandas dataframes. It either changes the max_columns or max_rows to infinite or resets those
    values to their default display limits.

    Args:
        dimension (string): Display dimension of a dataframe to alter; should be either "columns" or "rows".
        action (string): Action to be carried out on display settings; should be either "max" or "limit".
    """
    if dimension == "columns" and action == "max":
        pd.set_option('display.max_columns', None)
    elif dimension == "rows" and action == "max":
        pd.set_option('display.max_rows', None)
    elif dimension == "columns" and action == "limit":
        pd.reset_option('max_columns')
    else:
        pd.reset_option('max_rows')

In [3]:
adjust_df_display("columns", "max")
adjust_df_display("rows", "max")

In [10]:
## Constants
NET_X = 89.0
NET_Y = 0.0

# Shot types representing >= 1% of total shots. All others collapse to "other".
VALID_SHOT_TYPES = {"wrist", "snap", "slap", "tip-in", "backhand", "deflected"}

FEATURE_COLS = [
    # Positional
    "shot_distance",
    "shot_angle",
    "x_coord",
    "y_coord",
    "is_behind_net",
    # Situational
    "period",
    "shooting_team_strength_diff",
    "is_home_team",
    # Categorical
    "shot_type",
    "zone",
]

TARGET_COL = "is_goal"

In [11]:
## Helper functions
def _apply_filters(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove rows that should not be included in xG modeling:
      - Shots where coordinate normalization failed
      - Shootout events
      - Shots on an empty net (opponent's goalie is pulled)
    """
    # Only rows where coordinate normalization succeeded
    mask = df["coord_normalized"] == True

    # Exclude shootout shots
    mask &= df["period_type"] != "SO"

    # Exclude shots on an empty net
    # The opponent's net is empty when:
    # Shooting team is home AND away goalie is pulled
    # Shooting team is away AND home goalie is pulled
    opponent_goalie_pulled = (df["is_home_team"] & df["away_goalie_pulled"]) | (
        ~df["is_home_team"] & df["home_goalie_pulled"]
    )
    # exlude shots where oponent goalie is pulled
    mask &= ~opponent_goalie_pulled

    filtered = df[mask].copy()

    if len(filtered) == 0:
        raise ValueError("No rows remain after filtering — check input data.")

    return filtered


def _compute_geometry(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute shot geometry features.

    Assumes coordinates are normalized so the shooting team always attacks
    positive x, with the opponent net at (NET_X, NET_Y).

    Features added:
      shot_distance : Euclidean distance from shot location to net (feet)
      shot_angle    : Angle in degrees from the goal-line extended.
                      Shots in front of net → 0–90°
                      Shots behind the net  → 90–180°
      is_behind_net : True when x_coord > NET_X
    """
    # x-coordinate distance
    dx = NET_X - df["x_coord"]
    # y-coordinate distance
    dy = NET_Y - df["y_coord"]

    # Euclidean distance calculated using pythagorean theorem
    df["shot_distance"] = np.sqrt(dx**2 + dy**2)

    # arctan2(abs(y), dx) gives the angle from positive x-axis
    # dx is negative behind the net, so arctan2 naturally extends beyond 90°
    # shots directly in front of net are 0°
    # shots at the side are 90°
    # shots behind the net are greater than 90°
    df["shot_angle"] = np.degrees(np.arctan2(np.abs(df["y_coord"]), dx))

    df["is_behind_net"] = (df["x_coord"] > NET_X).astype(bool)

    return df


def _clean_shot_type(df: pd.DataFrame) -> pd.DataFrame:
    """
    Recode rare shot types (< 1% of total shots) and nulls to 'other'.

    Retained categories: wrist, snap, slap, tip-in, backhand, deflected.
    All others (wrap-around, poke, bat, between-legs, cradle, null) → 'other'.
    """
    df["shot_type"] = (
        df["shot_type"]
        .fillna("other")
        .apply(lambda x: x if x in VALID_SHOT_TYPES else "other")
    )
    return df


def _cast_types(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure downstream sklearn ColumnTransformer sees the right dtypes.
      - period cast to str so it is treated as categorical, not ordinal numeric.
      - is_home_team and is_behind_net cast to int (sklearn prefers 0/1 over bool).
    """
    df["period"] = df["period"].astype(str)
    df["is_home_team"] = df["is_home_team"].astype(int)
    df["is_behind_net"] = df["is_behind_net"].astype(int)
    return df

In [12]:
## Main function
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Main entry point for feature engineering.

    Takes the raw shots DataFrame (queried directly from the shots table)
    and returns a clean DataFrame of engineered features ready for input
    into an sklearn Pipeline.

    game_id is retained in the output so callers can perform season-based
    train/test splits without needing to re-join.

    Parameters
    ----------
    df : pd.DataFrame
        Raw shots DataFrame. Must contain all columns present in the shots table.

    Returns
    -------
    pd.DataFrame
        Columns: game_id, FEATURE_COLS, TARGET_COL.
        Filtered (no shootouts, no empty-net shots, coord_normalized only).
        Index reset.
    """
    df = _apply_filters(df)
    df = _compute_geometry(df)
    df = _clean_shot_type(df)
    df = _cast_types(df)

    output_cols = ["game_id"] + FEATURE_COLS + [TARGET_COL]
    return df[output_cols].reset_index(drop=True)

In [23]:
df = pd.read_csv('shots_100.csv')
df.head()

,game_id,event_id,sort_order,period,period_type,time_in_period,time_remaining,situation_code,strength_state,away_goalie_pulled,home_goalie_pulled,shooting_team_strength_state,shooting_team_strength_diff,shooter_id,goalie_id,shooter_team_id,shooter_team_abbrev,opponent_team_abbrev,is_home_team,event_type,x_coord_raw,y_coord_raw,x_coord,y_coord,coord_normalized,zone,shot_type,is_goal,miss_reason,created_at_utc,created_at_et
0,2019020001,10,11,1,REG,00:25,19:35,1551,5v5,False,False,5v5,0,8480801,8475883,9,OTT,TOR,False,goal,85.0,-1.0,85.0,-1.0,True,O,tip-in,True,NaN,2026-03-29 19:01:47.513943,2026-03-29 19:01:47.513943
1,2019020001,12,15,1,REG,00:38,19:22,1551,5v5,False,False,5v5,0,8479458,8475883,9,OTT,TOR,False,missed-shot,28.0,-37.0,28.0,-37.0,True,O,slap,False,wide-of-net,2026-03-29 19:01:47.513943,2026-03-29 19:01:47.513943
2,2019020001,15,28,1,REG,01:31,18:29,1451,4v5,False,False,5v4,1,8476853,8467950,10,TOR,OTT,True,shot-on-goal,-32.0,-2.0,32.0,-2.0,True,O,snap,False,NaN,2026-03-29 19:01:47.513943,2026-03-29 19:01:47.513943
3,2019020001,18,33,1,REG,01:58,18:02,1451,4v5,False,False,5v4,1,8478483,8467950,10,TOR,OTT,True,missed-shot,-46.0,-16.0,46.0,-16.0,True,O,wrist,False,wide-of-net,2026-03-29 19:01:47.513943,2026-03-29 19:01:47.513943
4,2019020001,19,44,1,REG,03:09,16:51,1551,5v5,False,False,5v5,0,8478857,8467950,10,TOR,OTT,True,missed-shot,-64.0,-4.0,64.0,-4.0,True,O,tip-in,False,wide-of-net,2026-03-29 19:01:47.513943,2026-03-29 19:01:47.513943


In [26]:
# df = _apply_filters(df)
# df = _compute_geometry(df)
# df = _clean_shot_type(df)
# df = _cast_types(df)
df = build_features(df)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 12 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   game_id                      100 non-null    int64  
 1   shot_distance                100 non-null    float64
 2   shot_angle                   100 non-null    float64
 3   x_coord                      100 non-null    float64
 4   y_coord                      100 non-null    float64
 5   is_behind_net                100 non-null    int64  
 6   period                       100 non-null    object 
 7   shooting_team_strength_diff  100 non-null    int64  
 8   is_home_team                 100 non-null    int64  
 9   shot_type                    100 non-null    object 
 10  zone                         100 non-null    object 
 11  is_goal                      100 non-null    bool   
dtypes: bool(1), float64(4), int64(4), object(3)
memory usage: 8.8+ KB


In [27]:
df

,game_id,shot_distance,shot_angle,x_coord,y_coord,is_behind_net,period,shooting_team_strength_diff,is_home_team,shot_type,zone,is_goal
0,2019020001,4.123106,14.036243,85.0,-1.0,0,1,0,0,tip-in,O,True
1,2019020001,71.344236,31.239215,28.0,-37.0,0,1,0,0,slap,O,False
2,2019020001,57.035077,2.009554,32.0,-2.0,0,1,1,1,snap,O,False
3,2019020001,45.880279,20.409883,46.0,-16.0,0,1,1,1,wrist,O,False
4,2019020001,25.317978,9.090277,64.0,-4.0,0,1,0,1,tip-in,O,False
5,2019020001,26.683328,12.994617,63.0,-6.0,0,1,0,0,snap,O,False
6,2019020001,36.055513,33.690068,59.0,-20.0,0,1,0,1,wrist,O,False
7,2019020001,5.000000,53.130102,86.0,4.0,0,1,0,1,tip-in,O,False
8,2019020001,55.226805,31.675469,42.0,-29.0,0,1,0,1,slap,O,False
9,2019020001,37.656341,10.713123,52.0,-7.0,0,1,0,1,slap,O,False


In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 34 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   game_id                       100 non-null    int64  
 1   event_id                      100 non-null    int64  
 2   sort_order                    100 non-null    int64  
 3   period                        100 non-null    object 
 4   period_type                   100 non-null    object 
 5   time_in_period                100 non-null    object 
 6   time_remaining                100 non-null    object 
 7   situation_code                100 non-null    int64  
 8   strength_state                100 non-null    object 
 9   away_goalie_pulled            100 non-null    bool   
 10  home_goalie_pulled            100 non-null    bool   
 11  shooting_team_strength_state  100 non-null    object 
 12  shooting_team_strength_diff   100 non-null    int64  
 13  shoote